# <font color="#418FDE" size="6.5" uppercase>**LeNet mit PyTorch**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Laden kleine Bilddatensätze mit torchvision und definieren Transformationen. 
- Implementieren ein LeNet-ähnliches CNN mit Conv2d, ReLU, Pooling und Dense-Schichten. 
- Trainieren und bewerten das CNN mit Validierung, Konfusionsmatrix und Fehlerbildern. 


## **1. torchvision Daten laden**

### **1.1. Daten laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_01.jpg?v=1784814494" width="250">



>* torchvision verbindet Bilddaten zuverlässig mit Training
>* Datensatzklassen liefern Bild-Label-Paare automatisch

>* Training und Test strikt trennen
>* Speicherort und Download reproduzierbar festlegen

>* DataLoader bündeln Bilder in effiziente Batches
>* Mischen fürs Training, feste Reihenfolge zum Testen



In [ ]:
#@title Python-Code - Daten laden

# Wir laden kleine Bilddaten für ein CNN.
# Transformationen machen Pixelwerte modellfreundlich und vergleichbar.
# Die Ausgabe zeigt Batchformen und Beispielbilder.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Der Digits-Datensatz ist klein und offline verfügbar.
digits = load_digits()
images = digits.images.astype(np.uint8)
labels = digits.target.astype(np.int64)

# Eine einfache Prüfung verhindert verwirrende Folgefehler.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden Graustufenbilder mit Form 8x8.")

# Diese Transformation ähnelt torchvision ToTensor und Normalize.
images_float = images.astype(np.float32) / 16.0
images_normalized = (images_float - 0.5) / 0.5

# CNNs erwarten oft die Form Batch, Kanal, Höhe, Breite.
images_for_cnn = images_normalized[:, np.newaxis, :, :]
batch_size = 8
batch_images = images_for_cnn[:batch_size]
batch_labels = labels[:batch_size]

print("Datensatz: sklearn digits, offline und klein")
print(f"Anzahl Bilder: {len(images)}")
print(f"Einzelbild vor Transformation: {images[0].shape}")
print(f"Batch für CNN: {batch_images.shape}")
print(f"Labels im ersten Batch: {batch_labels.tolist()}")

# Ein Bild macht die geladenen Daten direkt sichtbar.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(batch_images[0, 0], cmap="gray")
ax.set_title("Erstes geladenes Ziffernbild, normalisiert")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.2. Transformationen definieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_02.jpg?v=1784814496" width="250">



>* Bilder werden in passende Tensoren umgewandelt
>* Einheitliche Form verhindert Trainingsfehler

>* Normalisierung skaliert Pixelwerte für stabileres Lernen
>* CNNs erkennen Muster statt Helligkeitsunterschiede

>* Data Augmentation erzeugt sinnvolle Bildvarianten
>* Transformationen müssen zur Aufgabe passen



In [ ]:
#@title Python-Code - Transformationen definieren

# Wir definieren einfache Bildtransformationen für CNN-Eingaben.
# Das Beispiel zeigt Skalierung, Normalisierung und Kanalform.
# Am Ende vergleichen wir Rohbild und transformierte Werte.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Wir laden kleine Ziffernbilder direkt aus scikit-learn.
digits = load_digits()
image = digits.images[0]
label = int(digits.target[0])

# Diese Prüfung macht die erwartete Bildform sichtbar.
if image.shape != (8, 8):
    raise ValueError("Das Beispiel erwartet ein einzelnes 8x8-Bild.")

# Pixelwerte werden wie bei ToTensor in den Bereich null bis eins skaliert.
scaled_image = image.astype(np.float32) / 16.0

# Normalisierung verschiebt Werte um einen Mittelwert und skaliert sie.
mean_value = 0.5
std_value = 0.5
normalized_image = (scaled_image - mean_value) / std_value

# CNNs erwarten oft die Form Kanal, Höhe, Breite.
tensor_like_image = normalized_image[np.newaxis, :, :]

print(f"Datensatz: sklearn load_digits, Beispielklasse: {label}")
print(f"Rohbild-Form: {image.shape}, CNN-Form: {tensor_like_image.shape}")
print(f"Rohwerte: min={image.min():.0f}, max={image.max():.0f}")
print(f"Skaliert: min={scaled_image.min():.2f}, max={scaled_image.max():.2f}")
print(f"Normalisiert: min={normalized_image.min():.2f}, max={normalized_image.max():.2f}")

# Die Anzeige zeigt das kleine Eingabebild vor der Transformation.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray", vmin=0, vmax=16)
ax.set_title("Rohbild vor Tensor-Transformation")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.3. Conv Formen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_01_03.jpg?v=1784814498" width="250">



>* PyTorch nutzt Batch, Kanäle, Höhe, Breite
>* Falsche Reihenfolge verursacht häufig Modellprobleme

>* Filter erzeugen Merkmalskarten statt neuer Farben
>* Ausgabeformen müssen zur nächsten Schicht passen

>* Einheitliche Formen verbinden Transformationen und Architektur
>* CNNs lernen robuste räumliche Merkmale



In [ ]:
#@title Python-Code - Conv Formen verstehen

# Dieses Beispiel zeigt Formen vor einer Faltung.
# Wir nutzen kleine synthetische Graustufenbilder.
# Die Ausgabe erklärt Batch, Kanäle und Pixelgrößen.

import numpy as np
import matplotlib.pyplot as plt

# Ein Batch hat die Reihenfolge Bilder, Kanäle, Höhe, Breite.
batch_size = 4
channels = 1
height = 8
width = 8

# Die Werte bilden einfache synthetische Bildmuster.
images = np.zeros((batch_size, channels, height, width), dtype=np.float32)
images[0, 0, 2:6, 2:6] = 1.0
images[1, 0, :, 3:5] = 1.0
images[2, 0, 3:5, :] = 1.0
images[3, 0, np.arange(8), np.arange(8)] = 1.0

# Diese Prüfung macht die erwartete CNN-Form ausdrücklich sichtbar.
expected_shape = (4, 1, 8, 8)
if images.shape != expected_shape:
    raise ValueError("Die Bildform passt nicht zur erwarteten CNN-Eingabe.")

# Eine Conv2d-Schicht würde mehrere Filter als Ausgabekanäle erzeugen.
out_channels = 6
kernel_size = 3
stride = 1
padding = 0

# Ohne Padding schrumpfen Höhe und Breite um die Filterränder.
out_height = (height + 2 * padding - kernel_size) // stride + 1
out_width = (width + 2 * padding - kernel_size) // stride + 1
conv_shape = (batch_size, out_channels, out_height, out_width)

print(f"Eingabeform: {images.shape} = Batch, Kanäle, Höhe, Breite")
print(f"Filteranzahl: {out_channels} erzeugt {out_channels} Ausgabekanäle")
print(f"Ausgabeform nach Conv2d: {conv_shape}")
print(f"Für Matplotlib wird ein Bild als {images[0, 0].shape} angezeigt")

# Matplotlib zeigt nur Höhe und Breite eines einzelnen Kanals.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(images[0, 0], cmap="gray", vmin=0, vmax=1)
ax.set_title("Ein synthetisches 8x8-Graustufenbild")
ax.set_xlabel("Breite in Pixeln")
ax.set_ylabel("Höhe in Pixeln")
plt.show()



## **2. LeNet Modul**

### **2.1. Aktivierung und Pooling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_01.jpg?v=1784814487" width="250">



>* Aktivierungen ermöglichen komplexe Bildmuster
>* ReLU hebt starke Signale hervor

>* Max-Pooling behält die stärksten Merkmale
>* Kleinere Karten, robuster gegen Verschiebungen

>* Aktivierung und Pooling formen abstrakte Bildmerkmale
>* Pooling verdichtet Informationen mit bewusstem Detailverlust



In [ ]:
#@title Python-Code - Aktivierung und Pooling

# Dieses Beispiel zeigt Aktivierung und Pooling.
# ReLU entfernt negative Faltungsantworten aus Merkmalskarten.
# Max-Pooling verdichtet starke lokale Signale sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Ein kleines synthetisches Bild enthält eine helle Kante.
image = np.array(
    [[0, 0, 0, 0, 0, 0],
     [0, 0, 1, 1, 1, 0],
     [0, 0, 1, 1, 1, 0],
     [0, 0, 1, 1, 1, 0],
     [0, 0, 0, 0, 0, 0],
     [0, 0, 0, 0, 0, 0]],
    dtype=float,
)

# Dieser Filter reagiert stark auf vertikale Helligkeitswechsel.
kernel = np.array(
    [[-1, 0, 1],
     [-1, 0, 1],
     [-1, 0, 1]],
    dtype=float,
)

# Die Faltung erzeugt eine kleinere Merkmalskarte.
feature_map = np.zeros((4, 4), dtype=float)
for row in range(4):
    for col in range(4):
        patch = image[row:row + 3, col:col + 3]
        feature_map[row, col] = np.sum(patch * kernel)

# ReLU lässt positive Antworten stehen und setzt negative auf null.
relu_map = np.maximum(feature_map, 0)

# Max-Pooling nimmt pro Zweierblock nur den stärksten Wert.
pooled_map = np.zeros((2, 2), dtype=float)
for row in range(2):
    for col in range(2):
        block = relu_map[row * 2:row * 2 + 2, col * 2:col * 2 + 2]
        pooled_map[row, col] = np.max(block)

# Eine einfache Prüfung macht die Größenänderung nachvollziehbar.
if feature_map.shape != (4, 4) or pooled_map.shape != (2, 2):
    raise ValueError("Die erwarteten Merkmalskartenformen stimmen nicht.")

print("Eingabebild: 6x6 Werte")
print("Nach Faltung: 4x4 Merkmalskarte")
print("Nach Max-Pooling: 2x2 verdichtete Merkmalskarte")
print("Gepoolte Werte:", pooled_map.astype(int).ravel().tolist())

# Die Grafik zeigt die verdichtete Aktivierung nach ReLU und Pooling.
fig, ax = plt.subplots(figsize=(4, 4))
image_plot = ax.imshow(pooled_map, cmap="viridis", vmin=0, vmax=3)
ax.set_title("Max-Pooling nach ReLU")
ax.set_xlabel("Pooling-Spalte")
ax.set_ylabel("Pooling-Zeile")
fig.colorbar(image_plot, ax=ax, label="Aktivierungsstärke")
plt.show()



### **2.2. Flatten verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_02.jpg?v=1784814490" width="250">



>* CNNs erzeugen räumliche Merkmalskarten aus Bildern
>* Flatten macht daraus Vektoren für Dense-Schichten

>* Flatten macht Merkmalskarten zu einer Zahlenreihe
>* Keine neuen Parameter, nur Umformung für Dense

>* Flatten-Größe muss zur Dense-Schicht passen
>* Brücke von Merkmalen zur Klassifikation



In [ ]:
#@title Python-Code - Flatten verstehen

# Dieses Beispiel zeigt Flatten mit kleinen Merkmalskarten.
# Flatten verbindet räumliche CNN-Ausgaben mit Dense-Schichten.
# Die Ausgabe bestätigt passende Vektor- und Gewichtsdimensionen.

import numpy as np

# Wir bauen zwei künstliche CNN-Ausgaben im Format Batch, Kanal, Höhe, Breite.
feature_maps = np.arange(2 * 3 * 4 * 4).reshape(2, 3, 4, 4)

# Diese Prüfung macht die erwartete vierdimensionale Struktur ausdrücklich sichtbar.
if feature_maps.shape != (2, 3, 4, 4):
    raise ValueError("Die Merkmalskarten haben nicht die erwartete Form.")

# Flatten behält die Batch-Achse und legt alle Bildmerkmale nebeneinander.
batch_size = feature_maps.shape[0]
flattened = feature_maps.reshape(batch_size, -1)

# Die erste Dense-Schicht braucht genau so viele Eingaben pro Beispiel.
input_features = flattened.shape[1]
dense_weights = np.ones((input_features, 5))

# Eine Matrixmultiplikation simuliert die Weitergabe an fünf Dense-Neuronen.
dense_output = flattened @ dense_weights

print("Form vor Flatten: " + str(feature_maps.shape))
print("Form nach Flatten: " + str(flattened.shape))
print("Erwartete Dense-Eingaben: " + str(input_features))
print("Gewichtsform der Dense-Schicht: " + str(dense_weights.shape))
print("Ausgabeform nach Dense-Schicht: " + str(dense_output.shape))
print("Erste 10 Werte des ersten flachen Vektors: " + str(flattened[0, :10].tolist()))



### **2.3. LeNet Klasse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_02_03.jpg?v=1784814492" width="250">



>* LeNet bündelt CNN-Schichten als wiederverwendbares Modul
>* Klarer Datenfluss erzeugt nachvollziehbare Klassenvorhersagen

>* Faltungen erkennen einfache lokale Bildmuster
>* ReLU und Pooling verdichten robuste Merkmale

>* Flatten wandelt Merkmalskarten in Vektoren um
>* Dense-Schichten kombinieren Merkmale zur Klassenvorhersage



In [ ]:
#@title Python-Code - LeNet Klasse

# Dieses Beispiel baut eine kleine LeNet-Klasse.
# Die Schichten zeigen den typischen Datenfluss.
# Die Ausgabe bestätigt alle wichtigen Tensorformen.

import numpy as np

# Diese Klasse imitiert LeNet ohne Deep-Learning-Bibliothek.
class TinyLeNet:
    def __init__(self):
        self.conv1_filters = np.ones((2, 3, 3), dtype=float) / 9.0
        self.conv2_filters = np.ones((3, 2, 3, 3), dtype=float) / 9.0

        self.fc1_weights = np.full((12, 8), 0.05, dtype=float)
        self.fc2_weights = np.full((8, 3), 0.05, dtype=float)
        self.fc1_bias = np.zeros(8, dtype=float)
        self.fc2_bias = np.zeros(3, dtype=float)

    def relu(self, values):
        return np.maximum(values, 0.0)

    def conv2d_single(self, image, filters):
        out_channels, kernel_height, kernel_width = filters.shape
        output_height = image.shape[0] - kernel_height + 1
        output_width = image.shape[1] - kernel_width + 1

        output = np.zeros((out_channels, output_height, output_width))
        for channel in range(out_channels):
            for row in range(output_height):
                for col in range(output_width):
                    patch = image[row:row + kernel_height, col:col + kernel_width]
                    output[channel, row, col] = np.sum(patch * filters[channel])
        return output

    def conv2d_multi(self, feature_maps, filters):
        out_channels = filters.shape[0]
        output_height = feature_maps.shape[1] - filters.shape[2] + 1
        output_width = feature_maps.shape[2] - filters.shape[3] + 1

        output = np.zeros((out_channels, output_height, output_width))
        for out_channel in range(out_channels):
            for in_channel in range(feature_maps.shape[0]):
                current_filter = filters[out_channel, in_channel]
                current_map = feature_maps[in_channel]
                output[out_channel] += self.conv2d_single(
                    current_map,
                    current_filter.reshape(1, 3, 3),
                )[0]
        return output

    def max_pool2d(self, feature_maps):
        channels, height, width = feature_maps.shape
        output = np.zeros((channels, (height + 1) // 2, (width + 1) // 2))

        for channel in range(channels):
            for row in range(0, height, 2):
                for col in range(0, width, 2):
                    patch = feature_maps[channel, row:row + 2, col:col + 2]
                    output[channel, row // 2, col // 2] = np.max(patch)
        return output

    def forward(self, image):
        shapes = []
        shapes.append(("Eingabebild", image.shape))

        x = self.relu(self.conv2d_single(image, self.conv1_filters))
        shapes.append(("Conv1 + ReLU", x.shape))

        x = self.max_pool2d(x)
        shapes.append(("Pooling1", x.shape))

        x = self.relu(self.conv2d_multi(x, self.conv2_filters))
        shapes.append(("Conv2 + ReLU", x.shape))

        x = self.max_pool2d(x)
        shapes.append(("Pooling2", x.shape))

        x = x.reshape(-1)
        shapes.append(("Flatten", x.shape))

        x = self.relu(x @ self.fc1_weights + self.fc1_bias)
        shapes.append(("Dense1 + ReLU", x.shape))

        logits = x @ self.fc2_weights + self.fc2_bias
        shapes.append(("Dense2 Logits", logits.shape))
        return logits, shapes

# Ein kleines synthetisches Graustufenbild ersetzt echte Trainingsdaten.
image = np.zeros((12, 12), dtype=float)
image[3:9, 4:8] = 1.0
image[5:7, 2:10] = 0.5

# Die Modellklasse bündelt Aufbau und Vorwärtslauf.
model = TinyLeNet()
logits, layer_shapes = model.forward(image)

# Diese Prüfung macht die erwartete Klassenzahl sichtbar.
if logits.shape != (3,):
    raise ValueError("Die letzte Dense-Schicht sollte drei Werte liefern.")

print("LeNet-ähnlicher Vorwärtslauf mit einem 12x12-Bild:")
for layer_name, shape in layer_shapes:
    print(f"{layer_name}: {shape}")
print(f"Vorhersagewerte: {np.round(logits, 3).tolist()}")



## **3. Training und Fehleranalyse**

### **3.1. Training auf CPU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_01.jpg?v=1784814500" width="250">



>* CPU-Training macht Lernschritte gut nachvollziehbar
>* Validierung zeigt Lernen oder bloßes Auswendiglernen

>* Plane Laufzeit, Modellgröße und Batch-Größe realistisch
>* Vergleiche Metriken und erkenne Trainingsprobleme

>* Training, Validierung und Test sauber trennen
>* Konfusionsmatrix und Fehlerbilder gezielt interpretieren



In [ ]:
#@title Python-Code - Training auf CPU

# Wir trainieren ein kleines Bildmodell auf CPU.
# Validierung und Fehleranalyse stehen im Mittelpunkt.
# Die Ausgabe zeigt Metriken und Verwechslungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Der Digits-Datensatz enthält kleine Graustufenbilder.
digits = load_digits()
images = digits.images
targets = digits.target

# Eine einfache Prüfung schützt vor falschen Annahmen.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8x8-Bilder.")

# Bilder werden für scikit-learn zu Merkmalsvektoren.
features = images.reshape(images.shape[0], -1)
labels = targets

# Die Aufteilung trennt Training und Validierung sauber.
X_train, X_valid, y_train, y_valid = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Dieses kleine Modell läuft schnell auf der CPU.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train_scaled, y_train)

# Validierungsdaten simulieren unbekannte Bilder.
train_predictions = model.predict(X_train_scaled)
valid_predictions = model.predict(X_valid_scaled)

# Genauigkeit zeigt den groben Lernerfolg.
train_accuracy = accuracy_score(y_train, train_predictions)
valid_accuracy = accuracy_score(y_valid, valid_predictions)

# Die Konfusionsmatrix zeigt typische Verwechslungen.
confusion = confusion_matrix(y_valid, valid_predictions)
wrong_indices = np.where(valid_predictions != y_valid)[0]

# Ein Fehlerbild macht die Analyse anschaulich.
first_wrong = int(wrong_indices[0])
wrong_image = X_valid[first_wrong].reshape(8, 8)
true_label = int(y_valid[first_wrong])
predicted_label = int(valid_predictions[first_wrong])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsgenauigkeit: {train_accuracy:.3f}")
print(f"Validierungsgenauigkeit: {valid_accuracy:.3f}")
print(f"Anzahl Fehlerbilder: {len(wrong_indices)}")
print(f"Beispiel: wahr={true_label}, vorhergesagt={predicted_label}")

# Das Diagramm zeigt ein falsch klassifiziertes Bild.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(wrong_image, cmap="gray_r")
ax.set_title("Fehlerbild aus der Validierung")
ax.set_xlabel(f"Vorhersage: {predicted_label}")
ax.set_ylabel(f"Wahre Klasse: {true_label}")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **3.2. Fehlerbilder verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_02.jpg?v=1784814502" width="250">



>* Fehlerbilder zeigen Probleme hinter Genauigkeitswerten
>* Sie offenbaren gelernte oder störende Bildmerkmale

>* Fehlerbilder erklären Verwechslungen der Konfusionsmatrix
>* Sie zeigen Datenprobleme oder Modellschwächen

>* Fehler nach Mustern und Sicherheit gruppieren
>* Grenzen erkennen und nächste Schritte planen



In [ ]:
#@title Python-Code - Fehlerbilder verstehen

# Wir untersuchen falsch klassifizierte Ziffernbilder.
# Fehlerbilder verbinden Kennzahlen mit konkreten Beispielen.
# Die Grafik zeigt sichere Modellfehler.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Der kleine Digits-Datensatz ist offline verfügbar.
digits = load_digits()
images = digits.images
labels = digits.target

# Wir prüfen die erwartete Bildform vor dem Training.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Die Ziffernbilder haben nicht die erwartete Form.")

# Für das Modell werden die Bilder zu Merkmalsvektoren.
features = images.reshape(images.shape[0], -1)
class_names = [str(name) for name in digits.target_names]

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_valid, y_train, y_valid, img_train, img_valid = train_test_split(
    features, labels, images, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung und Modell werden nur auf Trainingsdaten angepasst.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Wahrscheinlichkeiten zeigen, wie sicher das Modell entscheidet.
probabilities = model.predict_proba(X_valid)
predictions = model.predict(X_valid)
confidence = probabilities.max(axis=1)

# Wir wählen den sichersten falschen Fall als Fehlerbild.
wrong_indices = np.flatnonzero(predictions != y_valid)
if len(wrong_indices) == 0:
    raise ValueError("Dieses Modell machte hier keinen Validierungsfehler.")

best_wrong_index = wrong_indices[np.argmax(confidence[wrong_indices])]
wrong_image = img_valid[best_wrong_index]
true_label = class_names[y_valid[best_wrong_index]]
predicted_label = class_names[predictions[best_wrong_index]]
wrong_confidence = confidence[best_wrong_index]
accuracy = accuracy_score(y_valid, predictions)

print(f"scikit-learn-Version: {sklearn_version}")
print(f"Validierungsgenauigkeit: {accuracy:.3f}")
print(f"Gezeigtes Fehlerbild: wahr={true_label}, vorhergesagt={predicted_label}")
print(f"Modellsicherheit für die falsche Klasse: {wrong_confidence:.3f}")

# Das einzelne Fehlerbild macht die Verwechslung sichtbar.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(wrong_image, cmap="gray_r", interpolation="nearest")
ax.set_title("Sicher falsch klassifiziertes Fehlerbild")
ax.set_xlabel(f"Vorhersage: {predicted_label}")
ax.set_ylabel(f"Wahre Klasse: {true_label}")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **3.3. Frameworks im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_A/image_03_03.jpg?v=1784814504" width="250">



>* PyTorch macht Trainingsschritte transparent nachvollziehbar
>* Bewertung prüft Generalisierung statt Auswendiglernen

>* Keras ermöglicht schnelle, abstrakte Prototypen.
>* PyTorch bietet flexible, transparente Fehleranalyse.

>* Frameworkwahl hängt vom Anwendungskontext ab
>* Validierung und Fehleranalyse bleiben zentral



In [ ]:
#@title Python-Code - Frameworks im Vergleich

# Dieses Beispiel vergleicht Bewertungsabläufe verschiedener Frameworks.
# Eine kleine Bildklassifikation ersetzt ein schweres CNN.
# Die Konfusionsmatrix zeigt typische Fehlerquellen sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Wir nutzen kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
targets = digits.target

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if images.shape[0] != targets.shape[0]:
    raise ValueError("Bildanzahl und Zielwerte passen nicht zusammen.")

# Für scikit-learn werden Bilder zu Merkmalsvektoren abgeflacht.
features = images.reshape(images.shape[0], -1)
class_names = [str(label) for label in digits.target_names]

# Die Aufteilung trennt Training und unabhängige Bewertung.
X_train, X_test, y_train, y_test = train_test_split(
    features, targets, test_size=0.25, stratify=targets, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Dieses Modell steht für einen kompakten Trainingsablauf.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train_scaled, y_train)

# Vorhersagen ermöglichen Genauigkeit und Fehleranalyse.
predictions = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)
confusion = confusion_matrix(y_test, predictions)

# Die stärkste Verwechslung wird aus der Matrix gelesen.
confusion_without_diagonal = confusion.copy()
np.fill_diagonal(confusion_without_diagonal, 0)
worst_pair = np.unravel_index(
    np.argmax(confusion_without_diagonal), confusion_without_diagonal.shape
)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.3f}")
print(
    f"Häufigste Verwechslung: {worst_pair[0]} als {worst_pair[1]} "
    f"({confusion_without_diagonal[worst_pair]} Fälle)"
)

# Die Matrix zeigt, was reine Genauigkeit verdecken kann.
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(confusion, cmap="Blues")

# Achsenbeschriftungen verbinden Zahlen mit Klassen.
ax.set_title("Konfusionsmatrix für Ziffernbilder")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")

# Klassenmarken machen die Matrix leichter lesbar.
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)

# Zahlen in den Feldern unterstützen die Fehleranalyse.
for row in range(confusion.shape[0]):
    for col in range(confusion.shape[1]):
        ax.text(col, row, confusion[row, col], ha="center", va="center")

fig.colorbar(image, ax=ax, label="Anzahl")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**LeNet mit PyTorch**</font>


In this lecture, you learned to:
- Laden kleine Bilddatensätze mit torchvision und definieren Transformationen. 
- Implementieren ein LeNet-ähnliches CNN mit Conv2d, ReLU, Pooling und Dense-Schichten. 
- Trainieren und bewerten das CNN mit Validierung, Konfusionsmatrix und Fehlerbildern. 

In the next Lecture (Lecture B), we will go over 'Transfer mit PyTorch'